# Silver Layer
Clean and normalize the Bronze data. Deduplicate, cast types, and split into proper relational tables (3NF).

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

CATALOG_SCHEMA = "dev.electroflow_pipeline."

# source tables (bronze)
bronze_customers_table = CATALOG_SCHEMA + "bronze_customers"
bronze_products_table = CATALOG_SCHEMA + "bronze_products"
bronze_orders_table = CATALOG_SCHEMA + "bronze_orders"
bronze_payments_table = CATALOG_SCHEMA + "bronze_payments"
bronze_coupons_table = CATALOG_SCHEMA + "bronze_coupons"

# target tables (silver)
silver_customers_table = CATALOG_SCHEMA + "silver_customers"
silver_products_table = CATALOG_SCHEMA + "silver_products"
silver_orders_table = CATALOG_SCHEMA + "silver_orders"
silver_order_items_table = CATALOG_SCHEMA + "silver_order_items"
silver_payments_table = CATALOG_SCHEMA + "silver_payments"
silver_coupons_table = CATALOG_SCHEMA + "silver_coupons"

In [0]:
def transform_customers():
    df = spark.table(bronze_customers_table)

    # deduplicate, fix dates, fill missing phones, encode gender
    df_clean = df.dropDuplicates(["customer_id"])\
        .withColumn("join_date", F.coalesce(
            F.try_to_date("join_date", "yyyy-MM-dd"), 
            F.try_to_date("join_date", "MM/dd/yyyy")
        )) \
        .withColumn("phone_number", F.coalesce(F.col("phone_number"), F.lit("unknown")))\
        .withColumn("gender_int", 
            F.when(F.col("gender") == "male", 1)
             .when(F.col("gender") == "female", 2)
             .otherwise(0))

    df_clean.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(silver_customers_table)
    print("customers done")
    return df_clean

In [0]:
def transform_orders():
    df = spark.table(bronze_orders_table)

    # orders header — drop the nested items array
    df_orders = df.dropDuplicates(["order_id"]).drop("items")
    df_orders.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(silver_orders_table)

    # order items — explode the items array into rows
    df_items = df.select("order_id", F.explode("items").alias("item"))\
        .select(
            "order_id",
            F.col("item.product_id").alias("product_id"),
            F.col("item.quantity").alias("quantity")
        )
    df_items = df_items.dropDuplicates(["order_id", "product_id"])
    df_items.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(silver_order_items_table)

    print("orders + order items done")
    return df_orders, df_items

In [0]:
def transform_products():
    df = spark.table(bronze_products_table)

    # deduplicate, cast price to decimal
    df_clean = df.dropDuplicates(["product_id"]) \
        .withColumn("price", F.col("base_price").cast("decimal(18,2)"))

    df_clean.write.format("delta").mode("overwrite").saveAsTable(silver_products_table)
    print("products done")
    return df_clean

In [0]:
def transform_payments():
    df = spark.table(bronze_payments_table)

    # deduplicate, cast payment value to decimal
    df_clean = df.dropDuplicates(["payment_id"]) \
        .withColumn("payment_value", F.col("payment_value").cast("decimal(18,2)"))

    df_clean.write.format("delta").mode("overwrite").saveAsTable(silver_payments_table)
    print("payments done")
    return df_clean

In [0]:
def transform_coupons():
    df = spark.table(bronze_coupons_table)

    df_clean = df.dropDuplicates(["coupon_code"])

    df_clean.write.format("delta").mode("overwrite").saveAsTable(silver_coupons_table)
    print("coupons done")
    return df_clean

In [0]:
silver_customers = transform_customers()
silver_orders, silver_order_items = transform_orders()
silver_products = transform_products()
silver_payments = transform_payments()
silver_coupons = transform_coupons()
print("silver layer complete")